# Load data

In [ ]:
# Load in data
import pandas as pd
movies = pd.read_csv("merged_movie_rag.csv", encoding="utf-8")

# View result
movies.head()

,wiki_movie_id,plot,movie_name,release_date,bo_revenue,runtime,languages,countries,genres,character_actor,clean_plot
0,23890098,"Shlykov, a hard-working taxi driver and Lyosha...",Taxi Blues,1990-09-07,NaN,110.0,Russian,"France, Soviet Union, Russia","Drama, World cinema","Unknown Character: Natalia Koliakanova, Pyot...","Shlykov, a hard-working taxi driver and Lyosha..."
1,31186339,The nation of Panem consists of a wealthy Capi...,The Hunger Games,2012-03-12,686533290.0,142.0,English,United States of America,"Action/Adventure, Science Fiction, Action, Drama",Foxface: Jacqueline Emerson\nKatniss Everdeen...,The nation of Panem consists of a wealthy Capi...
2,20663735,Poovalli Induchoodan is sentenced for six yea...,Narasimham,2000,NaN,175.0,Malayalam,India,"Musical, Action, Drama, Bollywood",M.K. Menon: Thilakan\nUnknown Character: Sai...,Poovalli Induchoodan is sentenced for six year...
3,2231378,"The Lemon Drop Kid , a New York City swindler,...",The Lemon Drop Kid,1951-03-08,2300000.0,91.0,English,United States of America,"Screwball comedy, Comedy","Unknown Character: Jane Darwell, Bob Hope, ...","The Lemon Drop Kid , a New York City swindler,..."
4,595909,Seventh-day Adventist Church pastor Michael Ch...,A Cry in the Dark,1988-11-03,6908797.0,121.0,English,"United States of America, Australia, New Zealand","Crime Fiction, Drama, Docudrama, World cinema,...","Unknown Character: Frank Holden, Charles Dan...",Seventh-day Adventist Church pastor Michael Ch...


# Chunking Strategy
For this RAG project, we represented each movie using three types of chunks: **metadata**, **cast**, and **plot**. This allows the retrieval system to separately capture factual movie attributes, actor/character information, and narrative plot content.

- **Metadata chunks** include structured details such as title, release date, runtime, languages, countries, genres, and box office revenue.
- **Cast chunks** include character and actor information.
- **Plot chunks** contain the movie summary using a sentence-based chunking strategy.

## Determining How to Chunk Plots
A key design decision in this pipeline is how to split plot summaries into chunks. Rather than using a generic text splitter, we adopted a rule-based, sentence-level approach to better preserve semantic structure and narrative flow.

To inform this strategy, we analyzed the distribution of sentence counts across all plots. This allowed us to understand the typical length of a plot and identify longer outliers. Based on this analysis, we can define a chunking rule that maintains full context for shorter plots while breaking longer plots into smaller, coherent segments that are more suitable for embedding and retrieval.

In [ ]:
# Create a subset of the dataframe
plots = movies[['wiki_movie_id', 'clean_plot', 'movie_name']].copy()

In [ ]:
# Define function to get the number of sentences for each plot
# Import nltk sentence tokenizer
from nltk.tokenize import sent_tokenize 

# Define function
def get_num_sentences(text):
    """
    Count the number of sentences in a given text string using
    NLTK's sentence tokenizer.

    Parameters
    ----------
    text : str
        A cleaned movie plot summary.

    Returns
    -------
    int
        The number of sentences in the input text.
    """
    return len(sent_tokenize(text))

# Apply to plots dataset
plots['num_sentences'] = plots['clean_plot'].apply(get_num_sentences)

# View result
plots.head()

,wiki_movie_id,clean_plot,movie_name,num_sentences
0,23890098,"Shlykov, a hard-working taxi driver and Lyosha...",Taxi Blues,1
1,31186339,The nation of Panem consists of a wealthy Capi...,The Hunger Games,52
2,20663735,Poovalli Induchoodan is sentenced for six year...,Narasimham,26
3,2231378,"The Lemon Drop Kid , a New York City swindler,...",The Lemon Drop Kid,49
4,595909,Seventh-day Adventist Church pastor Michael Ch...,A Cry in the Dark,15


In [5]:
# Metrics
plots['num_sentences'].describe()

count    42303.000000
mean        15.671584
std         16.618027
min          1.000000
25%          4.000000
50%          9.000000
75%         23.000000
max        344.000000
Name: num_sentences, dtype: float64

### Chunking strategy for plots
The distribution of sentence counts shows that most plots are relatively short, with a median of 9 sentences and an average of about 16 sentences. This indicates that the majority of plots can be kept as a single chunk without exceeding typical token limits or losing coherence.

However, the upper range of the distribution shows variability, with some plots reaching over 300 sentences. The 75th percentile is around 23 sentences, meaning that only a smaller portion of plots are substantially longer.

Based on this, we will use a conditional chunking strategy. Plots with 20 sentences or fewer are kept as a single chunk to preserve context, while longer plots are split into 10-sentence chunks. 

### Define Sentence-Based Plot Chunking Function

This function applies the plot chunking strategy we previously determined. Rather than splitting text by a fixed number of characters, it chunks each movie plot using sentence boundaries so that each chunk remains more natural and semantically consistent. We hope to avoid unnecessary splitting for shorter plots while making longer plots more manageable for embedding, retrieval, and downstream question answering in the RAG pipeline.

In [ ]:
# Define function for chunking plot summaries
def chunk_plot(plot_text, max_sent_full=20, split_size=10):
    """
    Split a movie plot into sentence-based chunks.
    If the plot contains 20 sentences or fewer, the full plot is
    returned as a single chunk. If the plot is longer than that threshold, it
    is split into non-overlapping chunks of  10 sentences.

    Parameters
    ----------
    plot_text : str
        A cleaned movie plot summary.
    max_sent_full : int, default=20
        The maximum number of sentences allowed before the plot is split
        into multiple chunks.
    split_size : int, default=10
        The number of sentences per chunk for plots longer than
        max_sent_full.

    Returns
    -------
    list
        A list of sentence-based plot chunks with no overlap.
    """
    # Get sentences using NLTK
    sentences = sent_tokenize(plot_text)

    # Keep shorter plots as a single chunk
    if len(sentences) <= max_sent_full:
        return [" ".join(sentences)]

    # Split longer plots into non-overlapping sentence chunks
    chunks = []
    for i in range(0, len(sentences), split_size):
        chunk = " ".join(sentences[i:i + split_size])
        chunks.append(chunk)

    return chunks

### Define Metadata chunk function

In [ ]:
# Define function to create a metadata text block for each movie
def create_metadata_chunk(row):
    """
    This function converts selected structured metadata fields from a movie
    record into a readable text block. 

    Parameters
    ----------
    row : pandas.Series
        A row containing movie metadata fields.

    Returns
    -------
    str
        A formatted text block containing the movie's metadata.
    """
    return f"""
Movie Title: {row['movie_name']}
Wiki ID: {row['wiki_movie_id']}
Release Date: {row['release_date']}
Runtime: {row['runtime']}
Languages: {row['languages']}
Countries: {row['countries']}
Genres: {row['genres']}
Box Office Revenue: {row['bo_revenue']}
""".strip()

### Define Cast chunk function

In [ ]:
# Define function to create a cast text block for each movie
def create_cast_chunk(row):
    """
    This function converts cast-related information from a movie record into
    a readable text block.

    Parameters
    ----------
    row : pandas.Series
        A row containing movie and cast information.

    Returns
    -------
    str
        A formatted text block containing the movie's cast information.
    """
    return f"""
Movie Title: {row['movie_name']}
Wiki ID: {row['wiki_movie_id']}
Release Date: {row['release_date']}
Characters and Actors:
{row['character_actor']}
""".strip()

## Apply Chunking Functions Across the Dataset
After defining the chunking functions, we apply them to each movie in the dataset to create the final dataset for retrieval ready text chunks.

Each movie is processed individually so that its metadata, cast information, and plot summary can be transformed into separate chunk types. For every movie, we create one metadata chunk, one cast chunk, and one or more plot chunks depending on the length of the plot (**recall chunking strategy for plots**).

Each chunk is stored as a dictionary containing the movie ID, movie title, chunk type, a unique chunk ID, and the chunk text itself. Storing the data in this format allows it to be easily converted into a pandas dataframe and later used for embedding, indexing, and retrieval.

For plot chunks, indexing is used to ensure that each chunk has a unique identifier, which is important for tracking and retrieval when multiple chunks are generated from a single movie.

In [ ]:
# Initialize a list to collect all chunk records before converting to a dataframe
chunked_rows = []

# Iterate through each movie in the dataframe
for i, row in movies.iterrows():
    # Core movie information to include in chunks
    wiki_id = row["wiki_movie_id"]
    title = row["movie_name"]
    release_date = row["release_date"]

    # Create metadata chunk
    metadata_text = create_metadata_chunk(row)
    chunked_rows.append({
        "wiki_movie_id": wiki_id,
        "movie_name": title,
        "chunk_type": "metadata",
        "chunk_id": f"{wiki_id}_meta_0",
        "chunk_text": metadata_text
    })

    # Create cast chunk
    cast_text = create_cast_chunk(row)
    chunked_rows.append({
        "wiki_movie_id": wiki_id,
        "movie_name": title,
        "chunk_type": "cast",
        "chunk_id": f"{wiki_id}_cast_0",
        "chunk_text": cast_text
    })

    # Create plot chunks
    plot_chunks = chunk_plot(row["clean_plot"])

    # Iterate through plot chunks (if mutiple are present)
    for idx, chunk in enumerate(plot_chunks): # Use enumerate to assign a unique index to each plot chunk
        # Create f-string 
        chunk_text = f"Movie Title: {title}\nWiki ID: {wiki_id}\nRelease Date: {release_date}\nPlot Summary: {chunk}"
        chunked_rows.append({
            "wiki_movie_id": wiki_id,
            "movie_name": title,
            "chunk_type": "plot",
            "chunk_id": f"{wiki_id}_plot_{idx}",
            "chunk_text": chunk_text
        })

After generating all the chunks, we convert the list of dictionaries into a pandas DataFrame.

In [10]:
# Convert list into dataframe
chunked_movies = pd.DataFrame(chunked_rows)

In [11]:
# View result
chunked_movies.head()

,wiki_movie_id,movie_name,chunk_type,chunk_id,chunk_text
0,23890098,Taxi Blues,metadata,23890098_meta_0,Movie Title: Taxi Blues\nWiki ID: 23890098\nRe...
1,23890098,Taxi Blues,cast,23890098_cast_0,Movie Title: Taxi Blues\nWiki ID: 23890098\nRe...
2,23890098,Taxi Blues,plot,23890098_plot_0,Movie Title: Taxi Blues\nWiki ID: 23890098\nRe...
3,31186339,The Hunger Games,metadata,31186339_meta_0,Movie Title: The Hunger Games\nWiki ID: 311863...
4,31186339,The Hunger Games,cast,31186339_cast_0,Movie Title: The Hunger Games\nWiki ID: 311863...


In [12]:
# View cast chunk
chunk_check = chunked_movies['chunk_text'].to_list()
print(chunk_check[4])

Movie Title: The Hunger Games
Wiki ID: 31186339
Release Date: 2012-03-12
Characters and Actors:
Foxface:  Jacqueline Emerson
Katniss Everdeen:  Jennifer Lawrence
Peeta Mellark:  Josh Hutcherson
Effie Trinket:  Elizabeth Banks
Gale Hawthorne:  Liam Hemsworth
Haymitch Abernathy:  Woody Harrelson
Clove:  Isabelle Fuhrman
Caesar Flickerman:  Stanley Tucci
Primrose Everdeen:  Willow Shields
President Snow:  Donald Sutherland
Cato:  Alexander Ludwig
Cinna:  Lenny Kravitz
Seneca Crane:  Wes Bentley
Rue:  Amandla Stenberg
Glimmer:  Leven Rambin
Claudius Templesmith:  Toby Jones
Marvel:  Jack Quaid
Mrs. Everdeen:  Paula Malcomson
Thresh:  Dayo Okeniyi
Katniss' Father:  Phillip Troy Linger
Game Tech:  Eric Hennig


In [13]:
# Check metadata chunk
print(chunk_check[3])

Movie Title: The Hunger Games
Wiki ID: 31186339
Release Date: 2012-03-12
Runtime: 142.0
Languages: English
Countries: United States of America
Genres: Action/Adventure, Science Fiction, Action, Drama
Box Office Revenue: 686533290.0


In [14]:
# View plot chunk
print(chunk_check[5])

Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: The nation of Panem consists of a wealthy Capitol and twelve poorer districts. As punishment for a past rebellion, each district must provide a boy and girl between the ages of 12 and 18 selected by lottery for the annual Hunger Games. The tributes must fight to the death in an arena; the sole survivor is rewarded with fame and wealth. In her first Reaping, 12-year-old Primrose Everdeen is chosen from District 12. Her older sister Katniss volunteers to take her place. Peeta Mellark, a baker's son who once gave Katniss bread when she was starving, is the other District 12 tribute. Katniss and Peeta are taken to the Capitol, accompanied by their frequently drunk mentor, past victor Haymitch Abernathy. He warns them about the "Career" tributes who train intensively at special academies and almost always win. During a TV interview with Caesar Flickerman, Peeta unexpectedly reveals his love for Katniss. Sh

# Save to csv file

In [15]:
# Save to csv file
chunked_movies.to_csv("movie_rag_chunks.csv", index=False, encoding="utf-8")